In [ ]:
import pandas as pd
master_final=pd.read_csv('master_final.csv')

category_health=master_final.groupby(['year_month','product_category_name']).agg(total_revenue=('price','sum'),
                                                                  avg_sentiment=('sentiment_score','mean'),
                                                                  avg_rating=('score','mean'),
                                                                  negative_rate=('sentiment',lambda x: (x=='Negative').mean()),
                                                                  total_orders=('order_id','count')
                                                                  ).reset_index()
health=category_health.merge(complaint_analysis,on='product_category_name',suffixes=('_health','_complaint'))
volume_score = health['compplaint_count'] / health['compplaint_count'].max()

raw_score = (
    ((health['avg_sentiment'] + 1)/2 * 0.35) +
    (health['avg_rating']/5 * 0.35) +
    ((1 - health['negative_rate']) * 0.1) +
    (volume_score * 0.2)
) * 100

health['health_score'] = (
    (raw_score - raw_score.min()) /
    (raw_score.max() - raw_score.min())
) * 100

health=health.sort_values(by='health_score',ascending=False)
health.to_csv('category_health_table.csv')